# BERT
This notebook covers loading BERT, fine-tuning it for the Malo et al. (2014) classification task, and topic clustering with BERTopic.

## 1. Data Preprocessing

### 1.1 Loading the data

In [ ]:
with open('data/FinancialPhraseBank/Sentences_75Agree.txt', 'r') as f: 
    FPB_75_in = f.read() # FinancialPhraseBank ships subsets at different annotator-agreement levels
with open('data/FinancialPhraseBank/Sentences_AllAgree.txt', 'r') as f:
    FPB_100_in = f.read()

In [ ]:
FPB_75_in[:200]

'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .@neutral\nWith the new production plant the company would increase its cap'

In [ ]:
from string import punctuation

FPB_75_lines = FPB_75_in.split('\n')

FPB_75_data = []
FPB_75_label = []
tags = {'negative':0,'neutral':1,'positive':2}
for line in FPB_75_lines[:-1]:
    #line = line.lower()  (no need to lowercase for BERT)
    #data_line = line.split('@')[0]
    FPB_75_label.append(tags[line.split('@')[1]])
    #data_line = ''.join([c for c in data_line if c not in punctuation]) # (no need to strip punctuation)
    #data_l = data_line.split(' ') (keep the raw sentence, not a list of words)
    #data_l = [c for c in data_l if len(c)>0] # (this dataset space-separates punctuation, which would leave empty tokens)
    FPB_75_data.append(line.split('@')[0])
    

In [ ]:
print(FPB_75_data[0],'\n',FPB_75_label[0])

According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing . 
 1


In [ ]:
FPB_100_lines = FPB_100_in.split('\n')
FPB_100_data = []
FPB_100_label = []
tags = {'negative':0,'neutral':1,'positive':2}
for line in FPB_100_lines[:-1]:
    FPB_75_label.append(tags[line.split('@')[1]])
    FPB_75_data.append(line.split('@')[0])

In [ ]:
FPB_data_raw = FPB_75_data + FPB_100_data
FPB_label_raw = FPB_75_label + FPB_100_label

In [ ]:
import json

# save to a JSON file
with open("data/FinancialPhraseBank/labels.json", "w", encoding="utf-8") as json_file:
    json.dump(FPB_label_raw, json_file, ensure_ascii=False, indent=4)

In [ ]:
with open("data/FinancialPhraseBank/eng_sent.txt", "w", encoding="utf-8") as file:
    for sentence in FPB_data_raw:
        file.write(sentence + "\n")

### 1.2 Tokenizer

In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_scheduler

import evaluate

In [ ]:
# # download and save bert-base-uncased locally (run once)
# # download the bert-base tokenizer
# tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# tokenizer.save_pretrained("./model/bert-base-uncased-tokenizer")
# # download the bert-base model
# model = BertForSequenceClassification.from_pretrained("bert-base-uncased")
# model.save_pretrained("./model/bert-base-uncased-model")


# afterwards, load from the local path
tokenizer_path = "./model/bert-base-uncased-tokenizer"
model_path = "./model/bert-base-uncased-model"

tokenizer = BertTokenizer.from_pretrained(tokenizer_path)
model = BertForSequenceClassification.from_pretrained(model_path)


In [ ]:
# load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("./model/bert-base-uncased-tokenizer")

In [ ]:
# tokenize and encode the sentences with BERT's tokenizer
encodings = tokenizer(
    FPB_data_raw,                  # list of input sentences
    truncation=True,            # truncate sentences longer than max_length
    padding=True,               # pad shorter sentences to a uniform length
    max_length=128,             # maximum sentence length; the rest is truncated
    return_tensors="pt"         # return PyTorch tensors
)

# get the encoded input_ids and attention_mask
input_ids = encodings["input_ids"]               # encoded token ids (like the int ids used with word2vec)
attention_mask = encodings["attention_mask"]     # attention mask: distinguishes real tokens from padding
labels = torch.tensor(FPB_label_raw)                    # convert labels to a PyTorch tensor

# create the dataset and data loader
# wrap input_ids, attention_mask and labels in a TensorDataset
dataset = torch.utils.data.TensorDataset(input_ids, attention_mask, labels)

### 1.3 Train/Val/Test Split

In [ ]:
from torch.utils.data import random_split, DataLoader
# set the random seed
torch.manual_seed(42)

# split ratios
train_size = int(0.8 * len(dataset))  # 80% train
val_size = int(0.1 * len(dataset))    # 10% validation
test_size = len(dataset) - train_size - val_size  # remaining 10% test

# split the dataset by ratio
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
)

# create the data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)  # training loader
val_loader = DataLoader(val_dataset, batch_size=16, drop_last=True)                   # validation loader
test_loader = DataLoader(test_dataset, batch_size=16, drop_last=True)                 # test loader

# check the split
print(f"Train size: {len(train_dataset)}, Validation size: {len(val_dataset)}, Test size: {len(test_dataset)}")

Train size: 4573, Validation size: 571, Test size: 573


## 2. BERT Classification

### 2.1 Fine-tuning the full model

In [ ]:
epochs = 3
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # choose device: GPU (CUDA) if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load the local pretrained BERT, ignoring the mismatched classification head
model_local = BertForSequenceClassification.from_pretrained(
    "./model/bert-base-uncased-model",  # local model path
    num_labels=3,                       # number of labels (3-way classification)
    ignore_mismatched_sizes=True        # ignore size mismatches
)

optimizer = optim.AdamW(model_local.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 5  # assume 5 epochs
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)


In [ ]:
import time

# load the evaluation metric
metric = evaluate.load("accuracy")
model_local.to(device)

# training and validation
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")

    # record training start time
    start_time = time.time()

    # train the model
    model_local.train()  # set the model to training mode
    for batch in train_loader:
        # move the batch to GPU or CPU
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        
        # forward pass: compute loss and logits
        outputs = model_local(**batch)
        loss = outputs.loss          # loss for this batch
        logits = outputs.logits      # model predictions (logits)

        # backpropagation and parameter update
        optimizer.zero_grad()        # clear the optimizer gradients
        loss.backward()              # compute gradients
        optimizer.step()             # update model parameters
        scheduler.step()             # update the learning rate

    # record training end time
    end_time = time.time()
    train_time = end_time - start_time  # training time
    print(f"Training time: {train_time:.2f} seconds")

    # validate the model
    model_local.eval()  # set the model to eval mode

    # record validation start time
    start_time = time.time()

    for batch in val_loader:
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        with torch.no_grad():  # disable gradients to save memory and speed up
            outputs = model_local(**batch)

        logits = outputs.logits                # get the logits
        predictions = torch.argmax(logits, dim=-1)  # predicted class
        metric.add_batch(predictions=predictions, references=batch["labels"])  # update the metric

    # record validation end time
    end_time = time.time()
    val_time = end_time - start_time  # elapsed time
    print(f"Validation time: {val_time:.2f} seconds")

    # compute and print the accuracy
    val_accuracy = metric.compute()["accuracy"]
    print(f"Epoch {epoch + 1}: Validation Accuracy = {val_accuracy:.4f}")

Epoch 1/3
Training time: 303.85 seconds
Validation time: 10.96 seconds
Epoch 1: Validation Accuracy = 0.9768
Epoch 2/3
Training time: 2429.00 seconds
Validation time: 7.41 seconds
Epoch 2: Validation Accuracy = 0.9714
Epoch 3/3
Training time: 226.74 seconds
Validation time: 9.90 seconds
Epoch 3: Validation Accuracy = 0.9714


In [ ]:
model_local.eval()
for batch in test_loader:
    batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
    with torch.no_grad():  # disable gradients to save memory and speed up
        outputs = model_local(**batch)

    logits = outputs.logits                # get the logits
    predictions = torch.argmax(logits, dim=-1)  # predicted class
    metric.add_batch(predictions=predictions, references=batch["labels"])  # update the metric


# compute and print the accuracy
test_accuracy = metric.compute()["accuracy"]
print(f" Test Accuracy = {test_accuracy:.4f}")

 Test Accuracy = 0.9732


### 2.2 Fine-tuning the classification head only

In [ ]:
epochs = 5
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # choose device: GPU (CUDA) if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load the local pretrained BERT, ignoring the mismatched classification head
model_local_freeze = BertForSequenceClassification.from_pretrained(
    "./model/bert-base-uncased-model",  # local model path
    num_labels=3,                       # number of labels (3-way classification)
    ignore_mismatched_sizes=True        # ignore size mismatches
)

optimizer = optim.AdamW(model_local_freeze.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 5  # assume 5 epochs
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)


In [ ]:
import time

model_local_freeze.to(device)
# freeze BERT's parameters (train only the classification head)
for param in model_local_freeze.bert.parameters():
    param.requires_grad = False

# load the evaluation metric
metric = evaluate.load("accuracy")

# training and validation

for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")

    # record training start time
    start_time = time.time()

    # train the model
    model_local_freeze.train()  # set the model to training mode
    for batch in train_loader:
        # move the batch to GPU or CPU
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        
        # forward pass: compute loss and logits
        outputs = model_local_freeze(**batch)
        loss = outputs.loss          # loss for this batch
        logits = outputs.logits      # model predictions (logits)

        # backpropagation and parameter update
        optimizer.zero_grad()        # clear the optimizer gradients
        loss.backward()              # compute gradients
        optimizer.step()             # update model parameters
        scheduler.step()             # update the learning rate

    # record training end time
    end_time = time.time()
    train_time = end_time - start_time  # training time
    print(f"Training time: {train_time:.2f} seconds")

    # validate the model
    model_local_freeze.eval()  # set the model to eval mode

    # record validation start time
    start_time = time.time()

    for batch in val_loader:
        batch = {k: v.to(device) for k, v in zip(["input_ids", "attention_mask", "labels"], batch)}
        with torch.no_grad():  # disable gradients to save memory and speed up
            outputs = model_local_freeze(**batch)

        logits = outputs.logits                # get the logits
        predictions = torch.argmax(logits, dim=-1)  # predicted class
        metric.add_batch(predictions=predictions, references=batch["labels"])  # update the metric

    # record validation end time
    end_time = time.time()
    val_time = end_time - start_time  # elapsed time
    print(f"Validation time: {val_time:.2f} seconds")

    # compute and print the accuracy
    val_accuracy = metric.compute()["accuracy"]
    print(f"Epoch {epoch + 1}: Validation Accuracy = {val_accuracy:.4f}")

Epoch 1/5
Training time: 96.44 seconds
Validation time: 10.81 seconds
Epoch 1: Validation Accuracy = 0.6089
Epoch 2/5
Training time: 97.02 seconds
Validation time: 12.24 seconds
Epoch 2: Validation Accuracy = 0.6089
Epoch 3/5
Training time: 100.60 seconds
Validation time: 11.52 seconds
Epoch 3: Validation Accuracy = 0.6089
Epoch 4/5
Training time: 100.97 seconds
Validation time: 11.42 seconds
Epoch 4: Validation Accuracy = 0.6089
Epoch 5/5
Training time: 99.64 seconds
Validation time: 11.64 seconds
Epoch 5: Validation Accuracy = 0.6089


## 3. BERTopic

In [ ]:
with open('data/FinancialPhraseBank/eng_sent.txt', 'r', encoding='utf-8') as file:
    content = file.read()
sentences = content.split('\n')
len(sentences)

5718

In [ ]:
# all-MiniLM-L6-v2 is a lightweight embedding model for BERTopic, much faster than a full BERT
# download and save locally (run once)
from sentence_transformers import SentenceTransformer

# download and save the model
model = SentenceTransformer("all-MiniLM-L6-v2")  # download the model
model.save("./model/all-MiniLM-L6-v2")  # save the model locally
print("Model saved locally at ./models/all-MiniLM-L6-v2")

Model saved locally at ./models/all-MiniLM-L6-v2


In [ ]:

# load the local model
embedding_model = SentenceTransformer("./model/all-MiniLM-L6-v2")  # load the model from the local path
print("Model loaded successfully from local path.")

Model loaded successfully from local path.


In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# prepare the data
embedding_model.to(device)
# extract sentence embeddings with the SentenceTransformer
topic_model = BERTopic(
    embedding_model=embedding_model,  # use a custom SentenceTransformer to generate embeddings
    top_n_words=5,                   # number of keywords shown per topic (default 10)
    min_topic_size=20,               # minimum topic size, i.e. min docs per topic (default 10)
    nr_topics=30                   # cap the number of topics at 30 (int or "auto"; default None)
)

start_time = time.time()

# cluster with BERTopic
topics, probs = topic_model.fit_transform(sentences)

end_time = time.time()
dur_time = end_time - start_time  # elapsed time
print(f"Clustering time: {dur_time:.2f} seconds")


In [ ]:


# view the topic info
print("\nGenerated Topics:")
print(topic_model.get_topic_info())



Generated Topics:
    Topic  Count                                        Name  \
0      -1   1476                            -1_the_and_of_in   
1       0   1505                        0_eur_mn_profit_from   
2       1    583                        1_finland_and_in_the   
3       2    245                            2_the_be_will_on   
4       3    231                 3_nokia_mobile_and_networks   
5       4    198                4_shares_share_rights_voting   
6       5    197                       5_and_services_as_our   
7       6    185                  6_not_disclosed_no_details   
8       7    134              7_steel_cargotec_bridge_cranes   
9       8    118                     8_paper_tonnes_mill_000   
10      9    100                  9_beer_meat_liters_percent   
11     10     91   10_approximately_employs_employees_people   
12     11     86         11_board_directors_dividend_meeting   
13     12     59     12_finnair_passenger_airline_passengers   
14     13     53     